### Titanic: Machine Learning from Disaster

So this is my first run through of the titanic data prblem. Just so i can put a few of the techniques i know in 1 place. Credit to all those places i have gained bits and pieces along the way...
I have added extra unnecessary eda which is completely covered in the ydataprofiling - more as a repo store of methods...



So based on a few very large assumptions of the titanic...it was early 20th century. For the lifeboats it would have been "women and children first" so i am going to assume that this is the case - but will check naturally. I also assume that older people would also have had preference, and the crew (captains etc. military would have been sacrificed!) I also assume the upper classes would have had a larger chance of survival than the lower classes. 

Overview
The data has been split into two groups:
Train and test

| Variable | Definition | Key |
|----------|------------|-----|
| survival | Survival   | 0 = No, 1 = Yes |
| pclass   | Ticket class | 1 = 1st, 2 = 2nd, 3 = 3rd |
| sex      | Sex        | |
| Age      | Age in years | |
| sibsp    | # of siblings / spouses aboard the Titanic | |
| parch    | # of parents / children aboard the Titanic | |
| ticket   | Ticket number | |
| fare     | Passenger fare | |
| cabin    | Cabin number | |
| embarked | Port of Embarkation | C = Cherbourg, Q = Queenstown, S = Southampton |



In [ ]:
# data analysis and wrangling
import pandas as pd
import numpy as np
import random as rnd

# visualization
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

#from ydata_profiling import ProfileReport
# To ignore warinings
import warnings
warnings.filterwarnings('ignore')


In [ ]:
TRAINING_DATA_PATH = '/kaggle/input/titanic/train.csv'
TEST_DATA_PATH = '/kaggle/input/titanic/test.csv'

In [ ]:
train_df = pd.read_csv(TRAINING_DATA_PATH)
test_df = pd.read_csv(TEST_DATA_PATH)
combine = [train_df, test_df] # nice technique to be able to iterate later through train and test

#profile = ProfileReport(train_df, title='Pandas Profiling Report', explorative=True)



### Data Profiling

In [ ]:
#profile.to_notebook_iframe()

### Standard EDA technuques

In [ ]:
print(train_df.columns.values)


In [ ]:
train_df.head()

In [ ]:
train_df.tail()

In [ ]:
train_df.info()
print('_'*40)
test_df.info()

In [ ]:
train_df.describe()


In [ ]:
train_df.describe(include=['O']) # includes object types 

In [ ]:
# Print unique values in the 'Sex' column
#unique_sex_values = combined_df['Sex'].unique()
#print(unique_sex_values)

In [ ]:
#unique_embarked_values = combined_df['Embarked'].unique()
#print(unique_embarked_values)

In [ ]:
train_df[["Sex", "Survived"]].groupby(['Sex'], as_index=False).mean().sort_values(by='Survived', ascending=False)

As expected it wasnt good to be a man 

In [ ]:
g = sns.FacetGrid(train_df, col='Survived')
g.map(plt.hist, 'Age', bins=20)

So the children did pretty well as expected. My assumption of the older people was wrong.

In [ ]:
train_df[["Parch", "Survived"]].groupby(['Parch'], as_index=False).mean().sort_values(by='Survived', ascending=False)

In [ ]:
train_df[["SibSp", "Survived"]].groupby(['SibSp'], as_index=False).mean().sort_values(by='Survived', ascending=False)

so if you had 1 sibling or spouse you had a better chance of survival.

In the Titanic dataset, the “Embarked” column represents the port where each passenger boarded the ship: C for Cherbourg, Q for Queenstown, and S for Southampton. This column can impact the survival rate due to several factors:

Socioeconomic Status: Passengers boarding from different ports might have had varying socioeconomic statuses. For instance, Cherbourg © had a higher proportion of first-class passengers, who had better access to lifeboats

In [ ]:
# grid = sns.FacetGrid(train_df, col='Embarked')
grid = sns.FacetGrid(train_df, row='Embarked',  aspect=1.6)
grid.map(sns.pointplot, 'Pclass', 'Survived', 'Sex', palette='deep')
grid.add_legend()

Dropping ticket and cabin as they are not relevant according to all the sources i have read. plus there are a lot of missing values. I guess if they were in their cabins at the time it could have had a large impact but i am also guessing that the cabins at top were first class passengers so is probably covered by that value

In [ ]:
print("Before", train_df.shape, test_df.shape, combine[0].shape, combine[1].shape)

train_df = train_df.drop(['Ticket', 'Cabin'], axis=1)
test_df = test_df.drop(['Ticket', 'Cabin'], axis=1)

combine = [train_df, test_df]

"After", train_df.shape, test_df.shape, combine[0].shape, combine[1].shape

### Title

This is an idea i have often come across so i am shamelessly stealing it :-)

In [ ]:
for dataset in combine:
    dataset['Title'] = dataset.Name.str.extract(' ([A-Za-z]+)\.', expand=False)

pd.crosstab(train_df['Title'], train_df['Sex'])

In [ ]:
for dataset in combine:
    dataset['Title'] = dataset['Title'].replace(['Lady', 'Countess', 'Don', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    dataset['Title'] = dataset['Title'].replace(['Capt', 'Col', 'Major', 'Dr', 'Rev'], 'Sacrificed')

    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs')
    
train_df[['Title', 'Survived']].groupby(['Title'], as_index=False).mean()

So the fact of my assumption that upper class women would have done well is not a problem as that will be covered by the pclass column, so i can lump the countess etc into the rare category. I will just have a quick look at the military to see if that is a negative or positive

In [ ]:
# Filter the DataFrame for military titles
military_titles = ['Capt', 'Col', 'Major', 'Dr', 'Rev']
military_df = train_df[train_df['Title'].isin(military_titles)]

# Group by the 'Survived' column and calculate the mean survival rate
military_survival_rate = military_df[['Title', 'Survived']].groupby('Title').mean()

# Display the survival rates for military titles
print(military_survival_rate)

In [ ]:
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5, "Sacrificed": 6}
for dataset in combine:
    dataset['Title'] = dataset['Title'].map(title_mapping)
    dataset['Title'] = dataset['Title'].fillna(0)

train_df.head()

Can drop passenger id as it is not relevant and also the name now we have got the title

In [ ]:
train_df = train_df.drop(['Name', 'PassengerId'], axis=1)
test_df = test_df.drop(['Name'], axis=1)
combine = [train_df, test_df]
train_df.shape, test_df.shape

### Converting a categorical features

In [ ]:
for dataset in combine:
    dataset['Sex'] = dataset['Sex'].map( {'female': 1, 'male': 0} ).astype(int)

train_df.head()

### Age 
There are a lot of missing ages and i know that master is a title for children so i will add those in. Then i came across an idea for the median age per class to be used. Shamelessly stealing. The ages are then banded.

In [ ]:
# Convert the 'Age' column to numeric, forcing non-numeric values to NaN
train_df['Age'] = pd.to_numeric(train_df['Age'], errors='coerce')

# Find rows where 'Age' is NaN (these were non-numeric values)
non_numeric_ages = train_df[train_df['Age'].isna()]

# Display the rows with non-numeric 'Age' values
print(non_numeric_ages)

In [ ]:
# Find the total number of rows where 'Age' is NaN
total_nan_ages = train_df['Age'].isna().sum()

# Print the total number of NaN ages
print(total_nan_ages)

In [ ]:
# Filter rows where 'Age' is NaN and 'Title' is 'Master'
nan_age_master = train_df[(train_df['Age'].isna()) & (train_df['Title'] == 4)]

# Count the number of such rows
count_nan_age_master = nan_age_master.shape[0]

# Print the count
print(count_nan_age_master)

In [ ]:
# grid = sns.FacetGrid(train_df, col='Pclass', hue='Gender')
grid = sns.FacetGrid(train_df, row='Pclass', col='Sex',  aspect=1.6)
grid.map(plt.hist, 'Age', alpha=.5, bins=20)
grid.add_legend()

In [ ]:
# Filter rows where 'Title' is 'Master'
master_ages = train_df[train_df['Title'] == 4]['Age']

# Display the ages
print(master_ages)

In [ ]:
# Set the ages of rows where 'Age' is NaN and 'Title' is 'Master' to 10
train_df.loc[(train_df['Age'].isna()) & (train_df['Title'] == 4), 'Age'] = 10
test_df.loc[(test_df['Age'].isna()) & (test_df['Title'] == 4), 'Age'] = 10

# Verify the changes
print(train_df[(train_df['Age'] == 10) & (train_df['Title'] == 4)])

In [ ]:
guess_ages = np.zeros((2,3))
guess_ages

In [ ]:
for dataset in combine:
    for i in range(0, 2):
        for j in range(0, 3):
            guess_df = dataset[(dataset['Sex'] == i) & \
                                  (dataset['Pclass'] == j+1)]['Age'].dropna()

            # age_mean = guess_df.mean()
            # age_std = guess_df.std()
            # age_guess = rnd.uniform(age_mean - age_std, age_mean + age_std)

            age_guess = guess_df.median()

            # Convert random age float to nearest .5 age
            guess_ages[i,j] = int( age_guess/0.5 + 0.5 ) * 0.5
            
    for i in range(0, 2):
        for j in range(0, 3):
            dataset.loc[ (dataset.Age.isnull()) & (dataset.Sex == i) & (dataset.Pclass == j+1),\
                    'Age'] = guess_ages[i,j]

    dataset['Age'] = dataset['Age'].astype(int)

train_df.head()


In [ ]:
train_df['AgeBand'] = pd.cut(train_df['Age'], 5)
train_df[['AgeBand', 'Survived']].groupby(['AgeBand'], as_index=False).mean().sort_values(by='AgeBand', ascending=True)

In [ ]:
for dataset in combine:    
    dataset.loc[ dataset['Age'] <= 16, 'Age'] = 0
    dataset.loc[(dataset['Age'] > 16) & (dataset['Age'] <= 32), 'Age'] = 1
    dataset.loc[(dataset['Age'] > 32) & (dataset['Age'] <= 48), 'Age'] = 2
    dataset.loc[(dataset['Age'] > 48) & (dataset['Age'] <= 64), 'Age'] = 3
    dataset.loc[ dataset['Age'] > 64, 'Age']
train_df.head()

In [ ]:
train_df = train_df.drop(['AgeBand'], axis=1)
combine = [train_df, test_df]
train_df.head()

### Family
The family size is relevant apparently in so far as people on their own survived a lot less 

In [ ]:
for dataset in combine:
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1

train_df[['FamilySize', 'Survived']].groupby(['FamilySize'], as_index=False).mean().sort_values(by='Survived', ascending=False)

In [ ]:
for dataset in combine:
    dataset['IsAlone'] = 0
    dataset.loc[dataset['FamilySize'] == 1, 'IsAlone'] = 1

train_df[['IsAlone', 'Survived']].groupby(['IsAlone'], as_index=False).mean()

In [ ]:
train_df = train_df.drop(['Parch', 'SibSp', 'FamilySize'], axis=1)
test_df = test_df.drop(['Parch', 'SibSp', 'FamilySize'], axis=1)
combine = [train_df, test_df]

train_df.head()

### Embarked
a categorical value so i will convert to a number. There are only 2 missing values and everyone seems to fill with the mode so why not...seems as good a try as any

In [ ]:
freq_port = train_df.Embarked.dropna().mode()[0]
freq_port

In [ ]:
for dataset in combine:
    dataset['Embarked'] = dataset['Embarked'].fillna(freq_port)
    
train_df[['Embarked', 'Survived']].groupby(['Embarked'], as_index=False).mean().sort_values(by='Survived', ascending=False)

In [ ]:
for dataset in combine:
    dataset['Embarked'] = dataset['Embarked'].map( {'S': 0, 'C': 1, 'Q': 2} ).astype(int)

train_df.head()

### Fare 
this can be banded, the assumption is the higher class and higher fared passenger do better

In [ ]:
test_df['Fare'].fillna(test_df['Fare'].dropna().median(), inplace=True)
test_df.head()

In [ ]:
train_df['FareBand'] = pd.qcut(train_df['Fare'], 4)
train_df[['FareBand', 'Survived']].groupby(['FareBand'], as_index=False).mean().sort_values(by='FareBand', ascending=True)

In [ ]:
for dataset in combine:
    dataset.loc[ dataset['Fare'] <= 7.91, 'Fare'] = 0
    dataset.loc[(dataset['Fare'] > 7.91) & (dataset['Fare'] <= 14.454), 'Fare'] = 1
    dataset.loc[(dataset['Fare'] > 14.454) & (dataset['Fare'] <= 31), 'Fare']   = 2
    dataset.loc[ dataset['Fare'] > 31, 'Fare'] = 3
    dataset['Fare'] = dataset['Fare'].astype(int)

train_df = train_df.drop(['FareBand'], axis=1)
combine = [train_df, test_df]
    
train_df.head(10)

In [ ]:
test_df.head(10)

In [ ]:
X_train = train_df.drop("Survived", axis=1)
Y_train = train_df["Survived"]
X_test  = test_df.drop("PassengerId", axis=1).copy()
X_train.shape, Y_train.shape, X_test.shape

### Modelling

In [ ]:


logreg = LogisticRegression()
logreg.fit(X_train, Y_train)
Y_pred = logreg.predict(X_test)
acc_log = round(logreg.score(X_train, Y_train) * 100, 2)
acc_log

In [ ]:
coeff_df = pd.DataFrame(train_df.columns.delete(0))
coeff_df.columns = ['Feature']
coeff_df["Correlation"] = pd.Series(logreg.coef_[0])

coeff_df.sort_values(by='Correlation', ascending=False)

In [ ]:
# Support Vector Machines

svc = SVC()
svc.fit(X_train, Y_train)
Y_pred = svc.predict(X_test)
acc_svc = round(svc.score(X_train, Y_train) * 100, 2)
acc_svc

In [ ]:
knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(X_train, Y_train)
Y_pred = knn.predict(X_test)
acc_knn = round(knn.score(X_train, Y_train) * 100, 2)
acc_knn

In [ ]:
# Decision Tree

decision_tree = DecisionTreeClassifier()
decision_tree.fit(X_train, Y_train)
Y_pred = decision_tree.predict(X_test)
acc_decision_tree = round(decision_tree.score(X_train, Y_train) * 100, 2)
acc_decision_tree

In [ ]:
# Random Forest

random_forest = RandomForestClassifier(n_estimators=100)
random_forest.fit(X_train, Y_train)
Y_pred = random_forest.predict(X_test)
random_forest.score(X_train, Y_train)
acc_random_forest = round(random_forest.score(X_train, Y_train) * 100, 2)
acc_random_forest

In [ ]:
models = pd.DataFrame({
    'Model': ['Support Vector Machines', 'KNN', 'Logistic Regression', 
              'Random Forest', 
              'Decision Tree'],
    'Score': [acc_svc, acc_knn, acc_log, 
              acc_random_forest, acc_decision_tree]})
models.sort_values(by='Score', ascending=False)

### Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [4, 6, 8, 10, 12],
    'criterion': ['gini', 'entropy']
}

# Initialize the Random Forest model
rf = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV with the Random Forest model and the parameter grid
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                           cv=5, n_jobs=-1, verbose=2, scoring='accuracy')

# Define the features and target variable
X_train = train_df.drop(columns=['Survived'])
y_train = train_df['Survived']

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Retrieve the best parameters and the best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best parameters: {best_params}")
print(f"Best cross-validation score: {best_score}")

In [ ]:
# Initialize the Random Forest model with the best parameters
best_rf = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_features=best_params['max_features'],
    max_depth=best_params['max_depth'],
    criterion=best_params['criterion'],
    random_state=42
)

# Fit the model to the training data
best_rf.fit(X_train, y_train)

# Evaluate the model on the training data
improved_score = best_rf.score(X_train, y_train)

print(f"Improved training score with best parameters: {improved_score * 100:.2f}%")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the default Random Forest model
default_rf = RandomForestClassifier(random_state=42)

# Fit the default model to the training data
default_rf.fit(X_train, y_train)

# Evaluate the default model on the training data
default_score = default_rf.score(X_train, y_train)

# Print the default model's training score
print(f"Default Random Forest training score: {default_score * 100:.2f}%")

# Print the hyperparameter-tuned model's training score
print(f"Hyperparameter-tuned Random Forest training score: {improved_score * 100:.2f}%")

### trying a stacking model

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score


# Define the base models
rf = RandomForestClassifier(random_state=42)
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

# Define the stacking classifier
stacking_clf = StackingClassifier(
    estimators=[('rf', rf), ('xgb', xgb)],
    final_estimator=RandomForestClassifier(random_state=42)
)

# Fit the stacking classifier to the training data
stacking_clf.fit(X_train, y_train)

# Evaluate the model using cross-validation
cv_scores = cross_val_score(stacking_clf, X_train, y_train, cv=5, scoring='accuracy')
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean cross-validation score: {cv_scores.mean()}")

# Make predictions on the test data
test_predictions = stacking_clf.predict(X_test)

# Display the predictions
print("Test set predictions:")
print(test_predictions)

In [ ]:
# Initialize the default Random Forest model
default_rf = RandomForestClassifier(random_state=42)

# Fit the default model to the training data
default_rf.fit(X_train, Y_train)

# Make predictions on the test data
Y_pred = default_rf.predict(X_test)

# Create a DataFrame with the PassengerId and the predicted Survived values
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": Y_pred
})

# Save the DataFrame to a CSV file
#submission.to_csv('submission.csv', index=False)

print("Submission file created successfully!")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score

# Define the parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [4, 6, 8, 10, 12],
    'criterion': ['gini', 'entropy']
}

# Initialize the Random Forest model
rf = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV with the Random Forest model and the parameter grid
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='accuracy')

# Fit GridSearchCV to the training data
grid_search.fit(X_train, Y_train)

# Retrieve the best parameters and the best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best parameters: {best_params}")
print(f"Best cross-validation score: {best_score}")

# Initialize the Random Forest model with the best parameters
best_rf = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_features=best_params['max_features'],
    max_depth=best_params['max_depth'],
    criterion=best_params['criterion'],
    random_state=42
)

# Fit the model to the training data
best_rf.fit(X_train, Y_train)

# Evaluate the model on the training data
train_score = best_rf.score(X_train, Y_train)
print(f"Training score: {train_score * 100:.2f}%")

# Evaluate the model using cross-validation
cv_scores = cross_val_score(best_rf, X_train, Y_train, cv=5, scoring='accuracy')
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean cross-validation score: {cv_scores.mean() * 100:.2f}%")

In [ ]:
# Fit the best_rf model to the training data
best_rf.fit(X_train, y_train)

# Make predictions on the test data
Y_pred = best_rf.predict(X_test)

# Create a DataFrame with the PassengerId and the predicted Survived values
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": Y_pred
})

# Save the DataFrame to a CSV file
submission.to_csv('submission.csv', index=False)

print("Submission file created successfully!")